In [ ]:
# !pip install -q transformers accelerate sentencepiece sacremoses pandas scikit-learn

import os, re, gc, json, shutil, pickle, random
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer,
                          Trainer, TrainingArguments, EarlyStoppingCallback,
                          XLMRobertaForSequenceClassification, XLMRobertaModel, XLMRobertaTokenizerFast,
                          TrainerCallback, set_seed)
from transformers.modeling_outputs import SequenceClassifierOutput

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU. Training will be extremely slow. Enable GPU in Colab runtime settings.")

Device: cuda


In [ ]:
# =====================================================================
# CONFIG  (the only cell you normally edit)
# =====================================================================
URL_3K = "https://docs.google.com/spreadsheets/d/1I-2e28z7ACkOeuOGEHqfdxtCh38jgROM/edit?gid=270439370#gid=270439370"   # TRAIN: Google Sheet link (share: Anyone with the link = Viewer) OR a .xlsx/.csv file name

SMOKE_TEST = False        # True: use a tiny synthetic CSV (upload dataset/smoke_test_synthetic.csv) to test the code
SMOKE_PATH = "smoke_test_synthetic.csv"

THRESHOLD = 3.0           # Likert_Mean >= 3.0 -> Offensive(1)
MAX_LEN = 160             # check the "truncated" percentage printed later; raise to 192 if it is high
MAX_EPOCHS = 4            # fixed number of epochs; the best epoch (macro-F1 on validation) is kept
REGION_PARAM_LR = 3e-3    # learning rate of the NEW region parameters (must be much larger than the encoder LR, or they never move)
EARLY_STOP_PATIENCE = None   # None = OFF. (patience=3 stopped a run after epoch 4 BEFORE it started to learn: the epoch-1 all-"Offensive" model looked "best")
BATCH = 16
LR = 2e-5
SEEDS = [42]              # final paper: [42, 1, 2, 3, 4]
RUN_ABLATIONS = False     # False: only the two configs in RUN_NAMES below (add more names for more ablations)
CROSS_TEST_FILE = "https://docs.google.com/spreadsheets/d/1bjUGd7GDmE6gS5OmgWnh94J1T9U6X8vWAUafoSbRPVQ/edit?gid=0#gid=0"   # TEST: 300 sentences x 4 regions; None = skip
CROSS_FOLDS = 5           # None: hold-out (all cross-region sentences are TEST only)
                          # K (e.g. 5): k-fold over the cross-region SENTENCES: the model also TRAINS on the other folds' sentences
                          #   (all regional versions of one sentence always stay together in train OR test)
EXPECTED_3K_ROWS = None   # set e.g. 3000 to get a warning when rows are missing

CONFIGS = {   # name -> which context blocks are given to the model
    "RAMP26_full":        dict(use_syntax=True,  use_comet=True,  use_demographics=True,  shuffle_demographics=False),
    "text_only(vanilla)": dict(use_syntax=False, use_comet=False, use_demographics=False, shuffle_demographics=False),
    "text+syntax":        dict(use_syntax=True,  use_comet=False, use_demographics=False, shuffle_demographics=False),
    "text+comet":         dict(use_syntax=False, use_comet=True,  use_demographics=False, shuffle_demographics=False),
    "text+demographics":  dict(use_syntax=False, use_comet=False, use_demographics=True,  shuffle_demographics=False),
    "full-no_demographics": dict(use_syntax=True, use_comet=True, use_demographics=False, shuffle_demographics=False),
    "full+SHUFFLED_demo": dict(use_syntax=True,  use_comet=True,  use_demographics=True,  shuffle_demographics=True),
    # ---- REGION-AWARE MODELS: region is a STRUCTURED input (not text). grammar + COMET stay as text; Region/Age/Gender text tags are OFF.
    #   bias  : one learned shift of the logits per (data source, region)          -> smallest, most direct
    #   embed : a learned region vector concatenated to the sentence representation
    #   heads : a separate small classifier head per (data source, region)
    "RAMP26_regionbias":  dict(use_syntax=True, use_comet=True, use_demographics=False, shuffle_demographics=False, region_model="bias"),
    "RAMP26_regionembed": dict(use_syntax=True, use_comet=True, use_demographics=False, shuffle_demographics=False, region_model="embed"),
    "RAMP26_regionheads": dict(use_syntax=True, use_comet=True, use_demographics=False, shuffle_demographics=False, region_model="heads"),
    "RAMP26_regionbias+SHUFFLED": dict(use_syntax=True, use_comet=True, use_demographics=False, shuffle_demographics=True, region_model="bias"),
}
RUN_NAMES = ["text_only(vanilla)", "RAMP26_regionbias", "RAMP26_regionbias+SHUFFLED"]
SEEDS = [1, 2]
# region-aware try-out:  RUN_NAMES = ["text_only(vanilla)", "RAMP26_regionbias", "RAMP26_regionbias+SHUFFLED"]  (then "RAMP26_regionembed", "RAMP26_regionheads")
# extra ablations: add "full-no_demographics", "text+syntax" (grammar only), "text+comet" ... to RUN_NAMES (each = 5 more trainings)

In [ ]:
# =====================================================================
# 1. DATA LOADING  (no silent row dropping, no fillna)
# =====================================================================
def to_export_urls(src):
    """Turns a normal Google Sheets link into export links (csv for the tab in the link, xlsx, Drive download)."""
    m = re.search(r"docs\.google\.com/spreadsheets/d/([A-Za-z0-9_-]+)", str(src))
    if not m:
        return None
    sid = m.group(1); g = re.search(r"gid=(\d+)", str(src)); gid = g.group(1) if g else "0"
    return dict(csv=f"https://docs.google.com/spreadsheets/d/{sid}/export?format=csv&gid={gid}",
                xlsx=f"https://docs.google.com/spreadsheets/d/{sid}/export?format=xlsx",
                drive=f"https://drive.google.com/uc?export=download&id={sid}")

def read_csv_safely(src, name):
    urls = to_export_urls(src)
    if urls:
        df, errs = None, []
        for kind in ["csv", "xlsx", "drive"]:
            try:
                d_ = pd.read_csv(urls["csv"]) if kind == "csv" else pd.read_excel(urls[kind])
                d_.columns = d_.columns.astype(str).str.strip()
                if not ({"Comment_Text", "Comment"} & set(d_.columns)):
                    raise ValueError(f"no Comment_Text column (got {list(d_.columns)[:4]}) - probably not the data tab or access denied")
                df = d_; print(f"[ok] {name}: loaded through the Google '{kind}' export"); break
            except Exception as e:
                errs.append(f"{kind}: {type(e).__name__}: {str(e)[:90]}")
        if df is None:
            raise RuntimeError(f"Cannot read the Google Sheet for {name}.\n  " + "\n  ".join(errs) +
                "\n  Fix: Share > General access > 'Anyone with the link' = Viewer. If the file is an uploaded Excel file "
                "(shows an XLSX badge), use File > Save as Google Sheets and paste the NEW link. Check the gid (tab) in the link.")
        return df.rename(columns={"Comment_id": "Comment_ID", "comment_id": "Comment_ID"})
    if str(src).lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(src)                                 # Excel file (first sheet)
    else:
        try:
            df = pd.read_csv(src)                               # strict C parser
        except Exception as e:
            print(f"[!] strict parse failed for {name} ({type(e).__name__}); retrying with python engine")
            df = pd.read_csv(src, engine="python", on_bad_lines="warn")
    df.columns = df.columns.astype(str).str.strip()
    return df.rename(columns={"Comment_id": "Comment_ID", "comment_id": "Comment_ID"})

def prepare(df, name):
    c_col = "Comment_Text" if "Comment_Text" in df.columns else "Comment"
    df["Comment_Text"] = df[c_col].astype(str).str.strip()
    ann = [c for c in ["Annotator_1", "Annotator_2", "Annotator_3"] if c in df.columns]
    long_col = next((c for c in ["Annotator_Rating", "Rating", "Annotator_rating"] if c in df.columns), None)
    if long_col:                      # LONG FORMAT: one row = one annotator's own rating (nothing is averaged)
        df["Likert_Mean"] = pd.to_numeric(df[long_col], errors="coerce")
    elif ann:
        for c in ann:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["Likert_Mean"] = df[ann].mean(axis=1)
    elif "Likert_Mean" in df.columns:
        df["Likert_Mean"] = pd.to_numeric(df["Likert_Mean"], errors="coerce")
    else:
        raise ValueError(f"{name}: no rating columns found. Columns: {list(df.columns)}")
    n0 = len(df)
    df = df.dropna(subset=["Comment_Text", "Likert_Mean"]).reset_index(drop=True)
    if len(df) != n0:
        print(f"[!] {name}: dropped {n0-len(df)} rows with missing text/ratings")
    for c, d in [("Language", "Bengali"), ("Region", "Dhaka"), ("Age", "18-22"), ("Gender", "Male")]:
        if c not in df.columns:
            df[c] = d
        df[c] = df[c].astype(str).str.strip()
    # make spellings identical between train and test files (the model reads these as plain words)
    df["Language"] = df["Language"].replace({"Bangla": "Bengali", "bangla": "Bengali", "bengali": "Bengali"})
    REGION_FIX = {"chittagong": "Chattogram", "chattagram": "Chattogram", "barisal": "Barishal", "dacca": "Dhaka"}
    df["Region"] = df["Region"].map(lambda r: REGION_FIX.get(r.lower(), r.title()))
    df["Gender"] = df["Gender"].str.title()
    AGE_OK = {"18-22", "23-30", "30+"}
    bad_age = ~df["Age"].isin(AGE_OK)
    if bad_age.any():
        print(f"[!] {name}: {int(bad_age.sum())} rows have an Age outside {sorted(AGE_OK)} -> set to 'Unknown'. "
              f"Fix the source data (e.g. Excel drag-fill produced '23-43').")
        df.loc[bad_age, "Age"] = "Unknown"
    df["Label"] = (df["Likert_Mean"] >= THRESHOLD).astype(int)
    print(f"[ok] {name}: rows={len(df)} unique_texts={df.Comment_Text.nunique()} "
          f"labels={df.Label.value_counts().to_dict()} offensive={df.Label.mean():.3f}")
    print("     sample texts (check Bengali is NOT broken):")
    for t in df.Comment_Text.sample(3, random_state=0):
        print("      ", t)
    return df

if SMOKE_TEST:
    df3 = prepare(read_csv_safely(SMOKE_PATH, "smoke"), "SMOKE (synthetic, not real)")
else:
    df3 = prepare(read_csv_safely(URL_3K, "3K"), "3K train data")
    if EXPECTED_3K_ROWS and len(df3) != EXPECTED_3K_ROWS:
        print(f"[!] WARNING: expected {EXPECTED_3K_ROWS} rows, got {len(df3)}. Check you opened the right sheet.")
df3["Source"] = 0          # 0 = main training file, 1 = cross-region file (the two files used different annotator pools / rating habits)
print("Demographic values in training data:",
      {c: sorted(df3[c].unique().tolist())[:12] for c in ["Region", "Age", "Gender"]})

[ok] 3K: loaded through the Google 'csv' export
[ok] 3K train data: rows=3000 unique_texts=2997 labels={1: 2221, 0: 779} offensive=0.740
     sample texts (check Bengali is NOT broken):
       আমেরিকা ইউরুপ ব্যর্থ হয়েছে। বাংলাদেশের মানুষে লক ডাউন কি বুঝেনা ফুল লক ডাউন করলে খাওয়াবে কে সবাইকে ⁉️ বাংলাদেশ সরকার যা পারতাছে করতাছে গালি দিয়ে কোনো ফায়দা নাই। ইউরুপ আমেরিকা চোখ রাখো সব বুঝে ফেলবা। আমরা চায়না নয় সব বন্ধ করে ঘরে বসাইয়া খাবার মত ধনী দেশ।
       আরে ভাই এই মুভিগুলো হলে গিয়ে কারা দেখে একটু জানার খুব ইচ্ছা
       নাসির যার সাথে খেলে এর থেকে বড় খেলা আর আছে নাকি।
Demographic values in training data: {'Region': ['Barishal', 'Chattogram', 'Dhaka', 'Khulna', 'London', 'Rajshahi', 'Rangpur', 'Sylhet'], 'Age': ['18-22', '23-30', '30+'], 'Gender': ['Female', 'Male']}


In [ ]:
# =====================================================================
# 2. MODELS FOR CONTEXT (translation + COMET)
# =====================================================================
trans_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-bn-en")
trans_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-bn-en").to(DEVICE).eval()
comet_tok = AutoTokenizer.from_pretrained("mismayil/comet-bart-ai2")
comet_model = AutoModelForSeq2SeqLM.from_pretrained("mismayil/comet-bart-ai2").to(DEVICE).eval()

def has_bengali(s):
    return any(0x0980 <= ord(ch) <= 0x09FF for ch in s)

# ---- syntax parser (Bengali-script + simple romanised/English patterns) ----
LAT, LATE = r"(?<![A-Za-z])", r"(?![A-Za-z])"
SELF_BN = r"(আমি|আমাকে|আমারে|আমার|আমায়|নিজেই|নিজেকে|নিজের|নিজে)"
SELF_EN = LAT + r"(i|me|my|myself|ami|amake|amar|nijer|nije)" + LATE
HON_BN = r"(আপনি|আপনারা|আপনাকে|আপনার|আপনাদের)"
HON_EN = LAT + r"(apni|apnar|apnake|apnara)" + LATE
INT_BN = r"(তুই|তোরা|তোকে|তোর|তোদের)"
INT_EN = LAT + r"(tui|tor|toke|tora|toder)" + LATE
FAM_BN = r"(তুমি|তোমরা|তোমাকে|তোমার|তোমাদের)"
FAM_EN = LAT + r"(you|u|ur|yourself|tumi|tomar|tomake|tomra)" + LATE

def extract_syntax(text):
    t = str(text).lower()
    mood = "Interrogative" if "?" in t else ("Exclamatory" if "!" in t else "Declarative")
    if re.search(INT_BN, t) or re.search(INT_EN, t):
        return "Intimate", mood, "Second_Person"
    if re.search(HON_BN, t) or re.search(HON_EN, t):
        return "Honorific", mood, "Second_Person"
    if re.search(FAM_BN, t) or re.search(FAM_EN, t):
        return "Familiar", mood, "Second_Person"
    if re.search(SELF_BN, t) or re.search(SELF_EN, t):
        return "None", mood, "Self"
    return "None", mood, "Third_Person"

def clean_comet(out, relation):
    out = out.replace("[GEN]", "").strip()
    if relation in out:
        out = out.split(relation)[-1].strip()
    return out

@torch.no_grad()
def build_cache(texts, cache_file=None):
    cache = {}
    if cache_file and os.path.exists(cache_file):
        cache = pickle.load(open(cache_file, "rb"))
    todo = [t for t in dict.fromkeys(texts) if t not in cache]
    print(f"[*] context cache: {len(cache)} cached, {len(todo)} to compute")
    # 1) translation (Bengali-script texts only)
    en = {t: t for t in todo}
    bn = [t for t in todo if has_bengali(t)]
    for i in range(0, len(bn), 32):
        b = bn[i:i + 32]
        x = trans_tok(b, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
        y = trans_model.generate(**x, max_length=128)
        for t, g in zip(b, y):
            en[t] = trans_tok.decode(g, skip_special_tokens=True).strip()
    # 2) COMET (3 relations per text, batched across 16 texts)
    rels = ["xIntent", "xReact", "xAttr"]
    for i in range(0, len(todo), 16):
        chunk = todo[i:i + 16]
        prompts, meta = [], []
        for t in chunk:
            pron, mood, tgt = extract_syntax(t)
            ev = {"Second_Person": f"Person directs at someone: '{en[t]}'",
                  "Self": f"Person self-reflects: '{en[t]}'"}.get(tgt, f"Person mentions: '{en[t]}'")
            meta.append((t, pron, mood, tgt))
            prompts += [f"{ev} {r} [GEN]" for r in rels]
        x = comet_tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=64).to(DEVICE)
        y = comet_model.generate(**x, max_length=30, num_beams=2, no_repeat_ngram_size=2, early_stopping=True)
        dec = [comet_tok.decode(g, skip_special_tokens=True) for g in y]
        for j, (t, pron, mood, tgt) in enumerate(meta):
            o = [clean_comet(dec[3 * j + k], rels[k]) for k in range(3)]
            cache[t] = dict(pron=pron, mood=mood, tgt=tgt,
                            intent=o[0] if len(o[0]) > 1 else "communication",
                            react=o[1] if len(o[1]) > 1 else "reacts",
                            attr=o[2] if len(o[2]) > 1 else "expressive")
    if cache_file:
        pickle.dump(cache, open(cache_file, "wb"))
    return cache

cache3 = build_cache(df3.Comment_Text.tolist(), "ramp26_cache_train.pkl")
print("example context:", cache3[df3.Comment_Text.iloc[0]])   # inspect: no 'xIntent' words should remain

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/806k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.25M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

... (14 outputs left out)

Loading weights:   0%|          | 0/514 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[*] context cache: 0 cached, 2997 to compute
example context: {'pron': 'None', 'mood': 'Exclamatory', 'tgt': 'Self', 'intent': 'none', 'react': 'regretful', 'attr': 'regretful'}


In [ ]:
# =====================================================================
# 3. SPLIT, DATASET, TRAINER
# =====================================================================
def group_split(df, seed):
    """70/15/15, stratified, the same text never appears in two splits."""
    g = df.groupby("Comment_Text").agg(y=("Label", lambda x: int(x.mean() >= .5))).reset_index()
    tr, tmp = train_test_split(g, test_size=.30, stratify=g.y, random_state=seed)
    va, te = train_test_split(tmp, test_size=.50, stratify=tmp.y, random_state=seed)
    pick = lambda s: df[df.Comment_Text.isin(set(s.Comment_Text))].reset_index(drop=True)
    return pick(tr), pick(va), pick(te)

def shuffle_demo(df, seed):
    """Control experiment: break the link between demographics and text/label."""
    d = df.copy()
    perm = d[["Region", "Age", "Gender"]].sample(frac=1, random_state=seed).reset_index(drop=True)
    d[["Region", "Age", "Gender"]] = perm.values
    return d

tokenizer = XLMRobertaTokenizerFast.from_pretrained("xlm-roberta-base")

def make_pair(row, cache, cfg):
    c = cache[row["Comment_Text"]]
    a = f"[Lang: {row['Language']}] {row['Comment_Text']}"
    parts = []
    if cfg["use_syntax"]:
        parts.append(f"[Pronoun: {c['pron']}] [Mood: {c['mood']}] [Target: {c['tgt']}]")
    if cfg["use_demographics"]:
        parts.append(f"[Region: {row['Region']}] [Age: {row['Age']}] [Gender: {row['Gender']}]")
    if cfg["use_comet"]:
        parts.append(f"[Intent: {c['intent']}] [React: {c['react']}] [Attr: {c['attr']}]")
    return a, (" ".join(parts) if parts else "[No context]")

def ctx_ids(df):
    """context id = data_source * (number of regions) + region index. Region effects are learned PER SOURCE because the two files
    were rated by different annotator pools (offensive rate 74% vs 58%); this stops the model mixing 'source' with 'region'."""
    return [int(sr) * NR + REGION2ID.get(rg, REGION2ID["Unknown"]) for sr, rg in zip(df["Source"], df["Region"])]

class RAMPDataset(torch.utils.data.Dataset):
    def __init__(self, df, cache, cfg, name=""):
        pairs = [make_pair(r, cache, cfg) for _, r in df.iterrows()]
        self.pairs = pairs
        self.enc = tokenizer([p[0] for p in pairs], [p[1] for p in pairs],
                             truncation="longest_first", padding="max_length", max_length=MAX_LEN)
        lens = np.array(self.enc["attention_mask"]).sum(1)
        self.trunc = float((lens >= MAX_LEN).mean())
        self.labels = df["Label"].tolist()
        self.ctx = ctx_ids(df) if cfg.get("region_model") else None      # only the region-aware models receive it
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        d = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        d["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        if self.ctx is not None:
            d["ctx_id"] = torch.tensor(self.ctx[i], dtype=torch.long)
        return d

class RegionAwareXLMR(nn.Module):
    """XLM-R encoder + region/source context given as a STRUCTURED input (an integer id), not as text."""
    def __init__(self, n_ctx, mode="bias", emb=32, drop=0.1):
        super().__init__()
        self.mode = mode
        self.enc = XLMRobertaModel.from_pretrained("xlm-roberta-base", add_pooling_layer=False)
        H = self.enc.config.hidden_size
        self.drop = nn.Dropout(drop)
        if mode == "embed":
            self.ctx_emb = nn.Embedding(n_ctx, emb); nn.init.normal_(self.ctx_emb.weight, std=0.02)
            self.cls = nn.Sequential(nn.Linear(H + emb, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, 2))
        else:
            self.cls = nn.Linear(H, 2)
        if mode == "bias":
            self.ctx_bias = nn.Embedding(n_ctx, 2); nn.init.zeros_(self.ctx_bias.weight)
        if mode == "heads":
            self.ctx_w = nn.Parameter(torch.zeros(n_ctx, H, 2)); self.ctx_b = nn.Parameter(torch.zeros(n_ctx, 2))
    def forward(self, input_ids=None, attention_mask=None, ctx_id=None, labels=None, **kw):
        h = self.drop(self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0])
        if self.mode == "embed":
            logits = self.cls(torch.cat([h, self.ctx_emb(ctx_id)], -1))
        else:
            logits = self.cls(h)
            if self.mode == "bias":
                logits = logits + self.ctx_bias(ctx_id)
            elif self.mode == "heads":
                logits = logits + torch.einsum("bh,bhc->bc", h, self.ctx_w[ctx_id]) + self.ctx_b[ctx_id]
        return SequenceClassifierOutput(logits=logits)

class BestStateCallback(TrainerCallback):
    """Keeps the weights of the best epoch (validation macro-F1) in memory and restores them at the end
    (used for the custom region models, whose checkpoints we do not save to disk)."""
    def __init__(self): self.best, self.state = -1.0, None
    def on_evaluate(self, args, state, control, metrics=None, model=None, **kw):
        f = (metrics or {}).get("eval_macro_f1", -1.0)
        if model is not None and f > self.best:
            self.best = f; self.state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    def on_train_end(self, args, state, control, model=None, **kw):
        if model is not None and self.state is not None:
            model.load_state_dict(self.state); print(f"[best epoch restored: validation macro-F1 {self.best:.4f}]")

def make_trainer_class(weights, new_lr=None):
    class BalancedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            labels = inputs.pop("labels")
            out = model(**inputs)
            loss = nn.CrossEntropyLoss(weight=weights)(out.logits, labels)
            return (loss, out) if return_outputs else loss
        def create_optimizer(self, *a, **k):
            if new_lr is None:
                return super().create_optimizer(*a, **k)
            if self.optimizer is None:        # encoder: small LR; NEW region parameters: large LR (they start from zero)
                enc = [p for n, p in self.model.named_parameters() if n.startswith("enc.")]
                new = [p for n, p in self.model.named_parameters() if not n.startswith("enc.")]
                self.optimizer = torch.optim.AdamW(
                    [{"params": enc, "lr": self.args.learning_rate},
                     {"params": new, "lr": new_lr, "weight_decay": 0.0}], weight_decay=self.args.weight_decay)
            return self.optimizer
    return BalancedTrainer

def compute_metrics(p):
    preds = np.argmax(p.predictions, -1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="macro", zero_division=0)
    return dict(accuracy=accuracy_score(p.label_ids, preds), macro_f1=f1, macro_precision=pr,
                macro_recall=rc, pred_offensive_rate=float(preds.mean()))   # ~1.0/~0.0 => collapsed

def dissent_report(df, preds):
    """Long format only: how well does the model reproduce a rater's OWN label when it differs from the text's majority?"""
    d = df.copy(); d["pred"] = preds
    if (d.groupby("Comment_Text").Label.transform("size") > 1).mean() < 0.5:
        return                                           # wide format: nothing to report
    maj = d.groupby("Comment_Text").Label.transform(lambda x: int(x.mean() >= .5))
    dis = d.Label != maj
    print(f"  rows where the rater DISAGREES with the text's majority: {int(dis.sum())} ({dis.mean()*100:.1f}%)")
    if dis.sum():
        print(f"  accuracy on those minority-view rows: {(d.pred[dis] == d.Label[dis]).mean():.3f}"
              f"   (a majority-vote model scores 0.000 here by construction)")
        print(f"  accuracy on majority-view rows      : {(d.pred[~dis] == d.Label[~dis]).mean():.3f}")
    lab_diff = d.groupby("Comment_Text").Label.nunique().gt(1)
    mod_diff = d.groupby("Comment_Text").pred.nunique().gt(1)
    print(f"  texts where raters disagree: {int(lab_diff.sum())} | model also gives different answers to "
          f"different raters on {int((lab_diff & mod_diff).sum())} of them; "
          f"and on {int((~lab_diff & mod_diff).sum())} texts where raters agreed (should be low)")

def subgroup_report(df, preds):
    d = df.copy(); d["pred"] = preds
    for col in ["Region", "Age", "Gender"]:
        rows = []
        for k, g in d.groupby(col):
            if len(g) >= 20 and g.Label.nunique() > 1:
                rows.append((k, len(g), round(f1_score(g.Label, g.pred, average="macro"), 3)))
        print(f"  macro-F1 by {col}:", rows)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [ ]:
# =====================================================================
# 4. ONE TRAINING RUN
# =====================================================================
def run_one(cfg_name, cfg, seed, cross_fn=None, extra_train=None, cross_test=None, fold=0):
    set_seed(seed); random.seed(seed); np.random.seed(seed)
    tr, va, te = group_split(df3, seed)
    if extra_train is not None:                       # k-fold mode: add the other folds' cross-region rows to TRAINING
        tr = pd.concat([tr, extra_train.reset_index(drop=True)], ignore_index=True)
    if cfg["shuffle_demographics"]:
        tr, va, te = (shuffle_demo(x, seed) for x in (tr, va, te))
    dtr, dva, dte = (RAMPDataset(x, cache3, cfg) for x in (tr, va, te))
    print(f"\n===== {cfg_name} | seed {seed} | train/val/test texts = "
          f"{tr.Comment_Text.nunique()}/{va.Comment_Text.nunique()}/{te.Comment_Text.nunique()} "
          f"| truncated {dtr.trunc*100:.1f}% =====")
    print("input example A:", dtr.pairs[0][0]); print("input example B:", dtr.pairs[0][1])
    w = compute_class_weight("balanced", classes=np.array([0, 1]), y=tr.Label.values)
    print("class weights (Safe, Offensive):", w.round(2))
    region_mode = cfg.get("region_model")
    if region_mode:
        model = RegionAwareXLMR(2 * NR, region_mode).to(DEVICE)
    else:
        model = XLMRobertaForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=2).to(DEVICE)
    steps_per_epoch = int(np.ceil(len(dtr) / BATCH))
    out_dir = f"./ckpt_{cfg_name.replace('(', '').replace(')', '').replace('+', '_')}_{seed}"
    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=MAX_EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
        warmup_steps=int(0.1 * steps_per_epoch * MAX_EPOCHS), weight_decay=0.01,
        eval_strategy="epoch", metric_for_best_model="macro_f1", greater_is_better=True,
        **(dict(save_strategy="no") if region_mode else dict(save_strategy="epoch", save_total_limit=2, load_best_model_at_end=True)),
        fp16=(DEVICE == "cuda"), seed=seed, logging_steps=50, report_to="none")
    T = make_trainer_class(torch.tensor(w, dtype=torch.float).to(DEVICE), new_lr=REGION_PARAM_LR if region_mode else None)
    trainer = T(model=model, args=args, train_dataset=dtr, eval_dataset=dva,
                compute_metrics=compute_metrics,
                callbacks=([EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)] if EARLY_STOP_PATIENCE else [])
                          + ([BestStateCallback()] if region_mode else []))
    trainer.train()
    res = trainer.predict(dte)                      # TEST only; val was used for checkpoint selection
    preds = res.predictions.argmax(-1)
    m = {k.replace("test_", ""): v for k, v in res.metrics.items() if isinstance(v, float)}
    print("TEST:", {k: round(v, 4) for k, v in m.items() if k in
          ["accuracy", "macro_f1", "macro_precision", "macro_recall", "pred_offensive_rate"]})
    print("confusion matrix (rows=true Safe/Off, cols=pred):\n", confusion_matrix(res.label_ids, preds))
    collapsed = int(m["pred_offensive_rate"] > 0.97 or m["pred_offensive_rate"] < 0.03)
    if collapsed:
        print(f"!!! COLLAPSED RUN: {cfg_name} fold {fold} predicts almost one class (rate {m['pred_offensive_rate']:.3f}). "
              f"Its numbers (and its cross-region predictions) are NOT a fair result: rerun this fold with another seed and report it.")
    maj = max(np.mean(res.label_ids), 1 - np.mean(res.label_ids))
    print(f"majority-class accuracy on this test set: {maj:.3f}  (macro-F1 of that trivial model ~0.43)")
    subgroup_report(te, preds)
    dissent_report(te, preds)
    pd.DataFrame(dict(Comment_Text=te.Comment_Text, Region=te.Region, Age=te.Age, Gender=te.Gender,
                      y_true=res.label_ids, y_pred=preds)).to_csv(
        f"preds_{cfg_name.replace('(', '').replace(')', '').replace('+', '_')}_{seed}.csv",
        index=False, encoding="utf-8-sig")
    if cross_fn is not None:
        cross_fn(trainer, cfg, cfg_name, seed, cross_test, fold, extra_train)
    row = dict(config=cfg_name, seed=seed, fold=fold, collapsed=collapsed, **m)
    del trainer, model; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    shutil.rmtree(out_dir, ignore_errors=True)
    return row

In [ ]:
# =====================================================================
# 5. CROSS-REGION TEST: the held-out sentences, each rated by several regions (long format)
# =====================================================================
def swap_demo_within_text(d, seed):
    """Control: for every sentence, shuffle which Region/Age/Gender tag goes with which row."""
    rng = np.random.RandomState(seed); d = d.copy()
    for _, idx in d.groupby("Comment_Text").groups.items():
        idx = list(idx); perm = rng.permutation(len(idx))
        d.loc[idx, ["Region", "Age", "Gender"]] = d.loc[[idx[i] for i in perm], ["Region", "Age", "Gender"]].values
    return d

def cross_eval(predict_fn, dx, name="", seed=0):
    """predict_fn(df) -> 0/1 predictions. Reports accuracy with the TRUE tags vs with SWAPPED tags."""
    dx = dx.reset_index(drop=True)
    return cross_summary(dx, np.asarray(predict_fn(dx)), np.asarray(predict_fn(swap_demo_within_text(dx, seed))), name, seed)

def cross_summary(dx, p_true, p_sw, name="", seed=0, prob_true=None, prob_sw=None, prior_shift=None):
    def m(d, p):
        return dict(acc=accuracy_score(d.Label, p), f1=f1_score(d.Label, p, average="macro"))
    out = dict(config=name, seed=seed, n_rows=len(dx), n_sentences=dx.Comment_Text.nunique())
    t, s_ = m(dx, p_true), m(dx, p_sw)
    out.update(acc_true_tags=t["acc"], f1_true_tags=t["f1"], acc_swapped_tags=s_["acc"], f1_swapped_tags=s_["f1"])
    split = dx.groupby("Comment_Text").Label.transform("nunique") > 1          # regions disagree on this sentence
    maj = dx.groupby("Comment_Text").Label.transform(lambda x: int(x.mean() >= .5))
    mino = dx.Label != maj                                                      # this region's view differs from the majority
    out["n_split_sentences"] = int(dx[split].Comment_Text.nunique()); out["n_minority_rows"] = int(mino.sum())
    if split.any():
        out["acc_split_true"] = float((p_true[split.values] == dx.Label[split]).mean())
        out["acc_split_swapped"] = float((p_sw[split.values] == dx.Label[split]).mean())
    if mino.any():
        out["acc_minority_true"] = float((p_true[mino.values] == dx.Label[mino]).mean())
        out["acc_minority_swapped"] = float((p_sw[mino.values] == dx.Label[mino]).mean())
    d2 = dx.assign(pred=p_true)
    out["per_region_f1"] = {k: round(f1_score(g.Label, g.pred, average="macro"), 3) for k, g in d2.groupby("Region") if g.Label.nunique() > 1}
    if prob_true is not None:
        d3 = dx.assign(pt=prob_true, ps=prob_sw)
        out["mean_prob_offensive_by_region_true_tags"] = {k: round(float(v), 3) for k, v in d3.groupby("Region").pt.mean().items()}
        out["human_offensive_rate_by_region"] = {k: round(float(v), 3) for k, v in d3.groupby("Region").Label.mean().items()}
        out["mean_abs_prob_change_when_tags_swapped"] = float(np.abs(prob_true - prob_sw).mean())
        if prior_shift is not None:                      # text-model + per-region prior shift (no extra training)
            lg = lambda q: np.log(np.clip(q, 1e-4, 1 - 1e-4) / (1 - np.clip(q, 1e-4, 1 - 1e-4)))
            adj = ((lg(prob_true) + prior_shift) > 0).astype(int)
            out["acc_region_prior_shift"] = float(accuracy_score(dx.Label, adj))
            out["f1_region_prior_shift"] = float(f1_score(dx.Label, adj, average="macro"))
    out["model_flips_on_split"] = float((d2[split].groupby("Comment_Text").pred.nunique() > 1).mean()) if split.any() else float("nan")
    out["model_flips_on_agreed"] = float((d2[~split].groupby("Comment_Text").pred.nunique() > 1).mean())
    return out

dx = cache_x = None
cross_rows = []
if CROSS_TEST_FILE:
    dx = prepare(read_csv_safely(CROSS_TEST_FILE, "cross-region test"), "cross-region test")
    leak = set(dx.Comment_Text) & set(df3.Comment_Text)
    assert not leak, f"{len(leak)} test sentences are also in the training data - remove them from training!"
    dx["Source"] = 1
    cache_x = build_cache(dx.Comment_Text.tolist(), "ramp26_cache_cross.pkl")
    cache3.update(cache_x)          # cross-region sentences may also be used for training (k-fold mode)

REGION2ID = {r: i for i, r in enumerate(sorted(set(df3.Region) | (set(dx.Region) if dx is not None else set()) | {"Unknown"}))}
NR = len(REGION2ID); print("region ids:", REGION2ID)

cross_store, cross_final = {}, {}

def paired_bootstrap(dx, pa, pb, n_boot=2000, seed=0):
    """Cluster bootstrap over SENTENCES (the 4 regional rows of a sentence move together). Returns mean diff and 95% CI of A - B."""
    rng = np.random.RandomState(seed); y = dx.Label.values
    sid = pd.factorize(dx.Comment_Text)[0]; S = sid.max() + 1
    groups = [np.where(sid == i)[0] for i in range(S)]; da, df_ = [], []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.randint(0, S, S)])
        da.append(accuracy_score(y[idx], pa[idx]) - accuracy_score(y[idx], pb[idx]))
        df_.append(f1_score(y[idx], pa[idx], average="macro") - f1_score(y[idx], pb[idx], average="macro"))
    ci = lambda d: (round(float(np.mean(d)), 4), round(float(np.percentile(d, 2.5)), 4), round(float(np.percentile(d, 97.5)), 4))
    return dict(acc_diff_mean_lo_hi=ci(da), f1_diff_mean_lo_hi=ci(df_))
def softmax_p1(logits):
    z = logits - logits.max(-1, keepdims=True); e = np.exp(z); return (e / e.sum(-1, keepdims=True))[:, 1]

def cross_fn(trainer, cfg, name, seed, test_df, fold, extra_train=None):
    if test_df is None: return
    test_df = test_df.reset_index(drop=True)
    lf = lambda d_: trainer.predict(RAMPDataset(d_, cache3, cfg)).predictions
    lt, ls = lf(test_df), lf(swap_demo_within_text(test_df, seed))
    shift = None        # per-region log-odds shift from the cross-region TRAINING rows of THIS fold (labels only, no test info)
    if extra_train is not None and len(extra_train):
        lg = lambda q: np.log(np.clip(q, 1e-4, 1 - 1e-4) / (1 - np.clip(q, 1e-4, 1 - 1e-4)))
        rate = {k: float(v) for k, v in extra_train.groupby("Region").Label.mean().items()}; rall = float(extra_train.Label.mean())
        shift = test_df.Region.map(lambda r: lg(rate.get(r, rall)) - lg(rall)).values
    parts = cross_store.setdefault((name, seed), [])
    parts.append((test_df, lt.argmax(-1), ls.argmax(-1), softmax_p1(lt), softmax_p1(ls), shift))
    if len(parts) == (CROSS_FOLDS or 1):                       # all folds finished -> every cross sentence predicted once
        dall = pd.concat([x[0] for x in parts], ignore_index=True)
        shifts = [x[5] for x in parts]
        prior_shift = np.concatenate(shifts) if all(sh is not None for sh in shifts) else None
        r = cross_summary(dall, np.concatenate([x[1] for x in parts]), np.concatenate([x[2] for x in parts]), name, seed,
                          prob_true=np.concatenate([x[3] for x in parts]), prob_sw=np.concatenate([x[4] for x in parts]),
                          prior_shift=prior_shift)
        cross_rows.append(r)
        cross_final[(name, seed)] = (dall, np.concatenate([x[1] for x in parts]), np.concatenate([x[2] for x in parts]))
        print("\n--- CROSS-REGION TEST (all folds combined) ---")
        for k, v in r.items():
            print(f"  {k}: {round(v, 3) if isinstance(v, float) else v}")
        pd.DataFrame(cross_rows).to_csv("crossregion_results.csv", index=False)

[ok] cross-region test: loaded through the Google 'csv' export
[!] cross-region test: 426 rows have an Age outside ['18-22', '23-30', '30+'] -> set to 'Unknown'. Fix the source data (e.g. Excel drag-fill produced '23-43').
[ok] cross-region test: rows=1200 unique_texts=300 labels={1: 700, 0: 500} offensive=0.583
     sample texts (check Bengali is NOT broken):
       সালির চাওয়ালের নাটক চাওয়া আজ থেকে বন্ধ
       তুমি একটা খচ্চর, তবুও তোমাকে ভালোবাসি।
       বেইশ্যাকে জুতা মার
[*] context cache: 0 cached, 300 to compute
region ids: {'Barishal': 0, 'Chattogram': 1, 'Dhaka': 2, 'Khulna': 3, 'London': 4, 'Rajshahi': 5, 'Rangpur': 6, 'Sylhet': 7, 'Unknown': 8}


In [ ]:
# =====================================================================
# 6. RUN EVERYTHING + SUMMARY
# =====================================================================
fold_defs = [(None, dx, 0)]                                   # hold-out: cross-region sentences are test only
if dx is not None and CROSS_FOLDS:
    sents = np.array(sorted(dx.Comment_Text.unique())); np.random.RandomState(SEEDS[0]).shuffle(sents)
    fold_defs = []
    for k in range(CROSS_FOLDS):
        te_s = set(sents[k::CROSS_FOLDS])
        fold_defs.append((dx[~dx.Comment_Text.isin(te_s)], dx[dx.Comment_Text.isin(te_s)].reset_index(drop=True), k))
    print(f"[k-fold] {CROSS_FOLDS} folds over {len(sents)} cross-region sentences (every regional version of a sentence stays together)")
all_rows = []
for name in RUN_NAMES:
    for i, s in enumerate(SEEDS):
        for extra, ctest, k in fold_defs:
            all_rows.append(run_one(name, CONFIGS[name], s, cross_fn=cross_fn,
                                    extra_train=extra, cross_test=ctest, fold=k))
            pd.DataFrame(all_rows).to_csv("results_runs.csv", index=False)   # saved after every run

runs = pd.DataFrame(all_rows)
cols = ["accuracy", "macro_f1", "macro_precision", "macro_recall"]
summary = runs.groupby("config")[cols].agg(["mean", "std"]).round(4)
print("\n=== SUMMARY (test set; mean/std over seeds) ===")
print(summary)
summary.to_csv("results_summary.csv")
print("collapsed runs per config (must be 0 for a fair comparison):", runs.groupby("config").collapsed.sum().to_dict())
print("\nSaved: results_runs.csv, results_summary.csv, preds_*.csv, crossregion_results.csv")

if cross_rows:
    cr = pd.DataFrame(cross_rows)
    cols = [c for c in ["config", "acc_true_tags", "f1_true_tags", "acc_swapped_tags", "f1_swapped_tags", "acc_region_prior_shift",
                        "f1_region_prior_shift", "acc_split_true", "acc_minority_true", "mean_abs_prob_change_when_tags_swapped"] if c in cr.columns]
    print("\n=== CROSS-REGION TEST: 300 sentences x 4 regions, every sentence tested exactly once ===")
    print(cr[cols].round(3).to_string(index=False))
    if "human_offensive_rate_by_region" in cr.columns:
        print("\nhuman offensive rate by region  :", cr.human_offensive_rate_by_region.iloc[0])
        for _, r_ in cr.iterrows():
            print(f"model mean P(offensive) [{r_['config']}]:", r_["mean_prob_offensive_by_region_true_tags"])
    # paired tests (95% CI over sentences). If the interval contains 0 the difference is NOT reliable.
    for (n1, s1), (d1, a1, a1sw) in cross_final.items():
        print(f"\n[{n1}] true tags minus swapped tags (do the tags matter?):", paired_bootstrap(d1, a1, a1sw))
        for (n2, s2), (d2, a2, _) in cross_final.items():
            if n2 == "text_only(vanilla)" and n1 != n2 and s1 == s2:
                print(f"[{n1}] minus [text_only]:", paired_bootstrap(d1, a1, a2))

[k-fold] 5 folds over 300 cross-region sentences (every regional version of a sentence stays together)

===== text_only(vanilla) | seed 1 | train/val/test texts = 2337/450/450 | truncated 0.5% =====
input example A: [Lang: Bengali] তুই একটা চরম বলদ, মুখ বন্ধ করে বসে থাক!
input example B: [No context]
class weights (Safe, Offensive): [1.58 0.73]


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (205 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Pred Offensive Rate
1,0.703437,0.658532,0.766667,0.573136,0.735714,0.576230,0.933333
2,0.663652,0.660693,0.684444,0.621095,0.616692,0.634327,0.668889
3,0.571327,0.618621,0.755556,0.667061,0.677069,0.660199,0.775556
4,0.472815,0.653464,0.731111,0.677832,0.670000,0.696350,0.666667


[best epoch restored: validation macro-F1 0.6778]


TEST: {'accuracy': 0.7267, 'macro_f1': 0.6739, 'macro_precision': 0.6663, 'macro_recall': 0.6933, 'pred_offensive_rate': 0.6622}
confusion matrix (rows=true Safe/Off, cols=pred):
 [[ 73  44]
 [ 79 254]]
majority-class accuracy on this test set: 0.740  (macro-F1 of that trivial model ~0.43)
  macro-F1 by Region: [('Chattogram', 91, 0.732), ('Dhaka', 182, 0.655), ('Khulna', 30, 0.689), ('Rajshahi', 41, 0.584), ('Sylhet', 76, 0.661)]
  macro-F1 by Age: [('18-22', 197, 0.706), ('23-30', 183, 0.632), ('30+', 70, 0.66)]
  macro-F1 by Gender: [('Female', 203, 0.626), ('Male', 247, 0.706)]



--- CROSS-REGION TEST (all folds combined) ---
  config: RAMP26_regionbias+SHUFFLED
  seed: 2
  n_rows: 1200
  n_sentences: 300
  acc_true_tags: 0.671
  f1_true_tags: 0.662
  acc_swapped_tags: 0.674
  f1_swapped_tags: 0.665
  n_split_sentences: 112
  n_minority_rows: 148
  acc_split_true: 0.533
  acc_split_swapped: 0.542
  acc_minority_true: 0.378
  acc_minority_swapped: 0.392
  per_region_f1: {'Chattogram': 0.69, 'Dhaka': 0.601, 'Rangpur': 0.696, 'Sylhet': 0.659}
  mean_prob_offensive_by_region_true_tags: {'Chattogram': 0.543, 'Dhaka': 0.542, 'Rangpur': 0.556, 'Sylhet': 0.543}
  human_offensive_rate_by_region: {'Chattogram': 0.6, 'Dhaka': 0.497, 'Rangpur': 0.63, 'Sylhet': 0.607}
  mean_abs_prob_change_when_tags_swapped: 0.009
  acc_region_prior_shift: 0.669
  f1_region_prior_shift: 0.661
  model_flips_on_split: 0.045
  model_flips_on_agreed: 0.005

=== SUMMARY (test set; mean/std over seeds) ===
                           accuracy         macro_f1         macro_precision  \
         